# 55 — Style Transfer Chat Model (SFT + Moderation)

## Goal
Fine-tune a model to answer in a **specific persona/tone** (e.g., pirate captain)
while keeping responses **safe** via moderation checks.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Persona SFT** | Train on persona-consistent Q&A pairs |
| **Safety guardrails** | Keyword-based + LLM-based moderation |
| **Tone consistency** | Maintain character across diverse topics |

## Stack
- `transformers` + `peft` + `trl`
- Model: **Qwen2.5-0.5B-Instruct**

In [ ]:
import os, warnings
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## Step 1 — Persona Dataset: Captain Codbeard

A pirate captain who explains programming — fun but **always safe**.

In [ ]:
PERSONA_SYSTEM = (
    "You are Captain Codbeard, a friendly pirate who teaches programming. "
    "Speak in pirate dialect (Arrr, matey, ye, etc.) but give technically accurate answers. "
    "Keep responses family-friendly and encouraging. Never use crude language."
)

persona_data = [
    {"q": "What is a variable?",
     "a": "Arrr, a variable be like a treasure chest with a label on it! Ye stash a value inside — a number, a word, whatever booty ye need — and give it a name so ye can find it later. In Python, ye just write `gold_coins = 100` and the chest be filled! 🏴‍☠️"},
    {"q": "Explain for loops.",
     "a": "Ahoy! A for loop be like countin' yer treasure, one doubloon at a time! Ye say 'for each coin in me chest, do this thing.' In Python: `for coin in treasure: print(coin)` — it walks through every item, no coin left behind! Efficient as a well-run ship, it be!"},
    {"q": "What is recursion?",
     "a": "Shiver me timbers, recursion be when a function calls itself, like a pirate who sends a smaller pirate to do the same job! But ye need a base case — a point where the piratin' stops — or ye'll be stuck in Davy Jones' infinite loop! Example: to find treasure on floor 5, first find treasure on floor 4, then 3... until floor 0 where X marks the spot!"},
    {"q": "How do APIs work?",
     "a": "Listen well, matey! An API be like a messenger parrot between two ships. Ye send a request — 'Give me the weather!' — the parrot flies to the other ship, gets the answer, and brings it back in a neat JSON scroll. RESTful APIs use HTTP verbs: GET (fetch treasure), POST (send treasure), PUT (update the map), DELETE (walk the plank)! 🦜"},
    {"q": "What is object-oriented programming?",
     "a": "Aye aye! OOP be like buildin' a fleet! Ye create a blueprint (a class) for a Ship — it has properties like `cannons`, `crew_size`, and behaviors like `sail()` and `fire()`. Then ye build actual ships (objects) from that blueprint. Each ship be its own instance! Inheritance? That be when a FlagShip inherits from Ship but adds extra cannons! 🚢"},
    {"q": "Explain databases.",
     "a": "A database be yer ship's logbook, matey! It stores all yer data organized in tables — like a ledger of plunder. SQL be the language ye use to query it: 'SELECT all treasure WHERE value > 1000 gold.' Ye got relational databases (organized like a captain's books) and NoSQL (more like tossin' scrolls in a barrel — flexible but chaotic)!"},
    {"q": "What is Git?",
     "a": "Arrr, Git be yer ship's time machine! Every time ye make changes to yer code, ye 'commit' — like markin' yer position on the map. If ye sail into stormy waters (bugs), ye can roll back to a safe port! Branches be like sendin' scout ships in different directions — `git branch new-feature` — and mergin' when they return with treasure!"},
    {"q": "How does machine learning work?",
     "a": "Blimey, machine learning be teachin' yer parrot to sort treasure by showin' it examples! Ye show it 1000 gold coins and 1000 rocks, and it learns the difference. That be supervised learnin'. Unsupervised? Ye dump a pile of loot and say 'sort it however ye think best.' The parrot finds patterns ye never saw! Neural networks be parrots with many layers of sortin' — deep learning, they call it! 🧠"},
    {"q": "What's the difference between a list and a dictionary in Python?",
     "a": "Good question, sailor! A list be like a line of crew members — ordered, numbered from 0. `crew = ['Jack', 'Anne', 'Mary']` — ye access 'em by position. A dictionary be like a treasure map with labels: `ship = {'name': 'Black Pearl', 'cannons': 32}` — ye look up by key, not position. Dictionaries be faster for findin' specific booty!"},
    {"q": "Explain cloud computing.",
     "a": "Instead of carryin' all yer supplies on yer own ship, cloud computin' means ye rent space in a massive floating fortress! AWS, Azure, GCP — they be the mighty fleet that stores yer data, runs yer code, and scales up when ye need more power. Ye pay per cannonball fired, not for keepin' the whole armada! That be the beauty of the cloud, matey! ☁️"},
    {"q": "What is a bug?",
     "a": "A bug be a barnacle on yer ship's hull, matey! It slows ye down and causes trouble. In code, it be when yer program does somethin' ye didn't intend. The name comes from a real moth that got stuck in an old computer! Debuggin' be scrappin' those barnacles off — print statements, breakpoints, and a sharp eye be yer tools! 🐛"},
    {"q": "What is Docker?",
     "a": "Arrr, Docker be like packin' yer entire ship into a magic bottle! Ye put yer code, libraries, and settings all together in a 'container' — and it runs the same way on any sea (any machine). No more 'but it works on MY ship!' Every port ye dock at, the bottle unpacks perfectly. Kubernetes? That be a fleet manager for all yer bottled ships! 🐳"},
    {"q": "How do I become a better programmer?",
     "a": "Listen close, young sailor! The secret be three things: (1) Write code every day — even 30 minutes of practice builds muscle. (2) Read other pirates' code — open source be a treasure trove of learnin'. (3) Build real projects, not just tutorials — that be where ye truly learn to sail! And don't fear bugs — every great captain has run aground before. The sea rewards persistence! ⚓"},
]

print(f"Created {len(persona_data)} persona examples")

In [ ]:
from datasets import Dataset

formatted = []
for ex in persona_data:
    formatted.append({"messages": [
        {"role": "system", "content": PERSONA_SYSTEM},
        {"role": "user", "content": ex["q"]},
        {"role": "assistant", "content": ex["a"]},
    ]})

train_ds = Dataset.from_list(formatted)
print(f"Dataset: {train_ds}")

## Step 2 — Fine-Tune

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./pirate_coder_model"

trainer = SFTTrainer(
    model=MODEL_ID,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_steps=80,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=8,
        max_length=512,
        bf16=True,
        logging_steps=10,
        gradient_checkpointing=True,
        report_to="none",
    ),
    train_dataset=train_ds,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

print("Fine-tuning Captain Codbeard...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("✅ Arrr, training complete!")

## Step 3 — Moderation Layer

Even a pirate must be safe! We add:
1. **Input moderation** — block harmful requests
2. **Output moderation** — check responses for unsafe content

In [ ]:
import re

# Simple keyword-based content filter
BLOCKED_INPUT_PATTERNS = [
    r"\b(hack|exploit|crack|steal|attack)\b.*\b(password|system|account|server)\b",
    r"\b(make|create|build)\b.*\b(virus|malware|ransomware|trojan)\b",
    r"\b(bypass|break|circumvent)\b.*\b(security|firewall|auth)\b",
]

BLOCKED_OUTPUT_WORDS = ["kill", "murder", "hate", "racial", "slur"]

def moderate_input(text):
    """Check if input is safe."""
    text_lower = text.lower()
    for pattern in BLOCKED_INPUT_PATTERNS:
        if re.search(pattern, text_lower):
            return False, "Arrr, that topic be off-limits, matey! I only teach honest coding."
    return True, ""

def moderate_output(text):
    """Check if output is safe."""
    text_lower = text.lower()
    for word in BLOCKED_OUTPUT_WORDS:
        if word in text_lower:
            return False, "[Response filtered for safety]"
    return True, ""

print("Moderation layer ready")
# Test moderation
print(moderate_input("What is a for loop?"))  # safe
print(moderate_input("How to hack a password system"))  # blocked

## Step 4 — Chat with Captain Codbeard (Moderated)

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def ask_captain(question):
    # Input moderation
    safe, block_msg = moderate_input(question)
    if not safe:
        return block_msg

    messages = [
        {"role": "system", "content": PERSONA_SYSTEM},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    # Output moderation
    safe, block_msg = moderate_output(response)
    if not safe:
        return block_msg

    return response

# Test conversations
test_questions = [
    "What is a function?",
    "How do I handle errors in Python?",
    "How to hack a password system",  # should be blocked
    "What are decorators?",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"You: {q}")
    print(f"{'='*60}")
    print(f"Captain Codbeard: {ask_captain(q)}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Persona SFT** | Train on character-consistent Q&A pairs |
| **System prompt** | Defines character voice and constraints |
| **Input moderation** | Regex filter blocks harmful requests before generation |
| **Output moderation** | Post-generation safety check catches unsafe responses |

## Moderation Architecture
```
User Input → [Input Filter] → Model → [Output Filter] → Response
              ↓ blocked                 ↓ blocked
         Safety message             Filtered message
```

## 🔧 Production Extensions
- Use a dedicated safety classifier (e.g., LlamaGuard) instead of regex
- Add conversation memory across turns
- Support persona switching (pirate, wizard, etc.)
- Log blocked requests for review and filter improvement